# Két lemez közös éllel

A két síklemez közös Gmsh-élhez csatlakozik. A középfelület elmozdulása és a fizikai forgás két háromkomponensű globális mező.

In [13]:
using LowLevelFEM, LinearAlgebra#, SparseArrays

openGeometry("folded-plate.geo")

In [14]:
mat = Material("plates", E=2e5, ν=0.3)

U = Field([mat], type=:VectorField, dim=3, field=:u)
Φ = Field([mat], type=:VectorField, dim=3, field=:φ);

In [15]:
n = normalVector(U, "plates")
tξ, tη = tangentVectors(U, "plates");

In [16]:
n = [n[1], n[2], n[3]]
tξ = [tξ[1], tξ[2], tξ[3]]
tη = [tη[1], tη[2], tη[3]];

In [17]:
# kron(tξ, tη)' ⋅ Grad(U) gives tξᵀ(∇U)tη.

A_u = vcat(kron(tξ,tξ)',
           kron(tη,tη)',
           (kron(tξ,tη) + kron(tη,tξ))',
           zeros(3,9),
           kron(n,tη)',
           kron(n,tξ)')

A_φ = vcat(zeros(3,9),
           kron(tη,tξ)',
           -kron(tξ,tη)',
           (kron(tη,tη) - kron(tξ,tξ))',
           zeros(2,9))

G_φ = vcat(zeros(6,3), -tξ', tη');

In [18]:
b = 0.12
k_s = 4/5

C = D(:PlaneStress, mat)

D1 = b*C

D2 = b^3 / 12 * C

D3 = k_s * b * mat.μ * Matrix{Float64}(I, 2, 2)

B_mb = A_u[1:6, :] ⋅ Grad(U) + A_φ[1:6, :] ⋅ Grad(Φ)
B_s  = A_u[7:8, :] ⋅ Grad(U) + G_φ[7:8, :] ⋅ Φ
D_mb = cat(D1, D2; dims=(1, 2))

K = ∫(B_mb' ⋅ D_mb ⋅ B_mb; Γ="plates") +
    ∫(B_s'  ⋅ D3   ⋅ B_s;  Γ="plates", gauss="CompositeGauss1")

K[:,:]

396×396 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6498 stored entries:
⎡⠱⢆⠑⠂⠶⠤⠘⢢⣄⡀⠏⠤⠀⠀⠒⠆⠆⠀⢠⣀⠳⢌⠐⠢⠆⠤⠑⠠⢄⡀⠃⠄⠀⠐⠠⠂⠂⠀⠠⢀⎤
⎢⠱⠀⠻⣦⡀⠀⠨⠄⠀⠀⠙⠘⠸⠆⣦⣤⡀⠀⠀⠀⠥⠘⠷⣤⡀⠀⠡⠀⠀⠀⠃⠛⠰⢰⣄⣤⡀⠀⠀⠀⎥
⎢⠘⡇⠀⠈⠻⣦⡀⠀⠀⠀⢁⣤⢰⠸⠘⠋⠁⠀⠀⠀⢘⡀⠀⠙⢳⣄⠀⠀⠀⠀⢁⡄⡆⠰⠙⠈⠁⠀⠀⠀⎥
⎢⠲⣀⠂⠆⠀⠈⠻⣦⡀⠀⠛⠉⠀⠀⠀⠆⢶⢠⢠⡀⠒⡐⠀⠂⠀⠉⠳⣦⡀⠀⠛⠁⠀⠀⠀⠆⠆⢠⢠⠀⎥
⎢⠀⠹⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⡀⣀⢸⢻⠀⠱⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⡀⡀⣬⢳⎥
⎢⠋⡅⣓⠀⠁⣴⡟⠀⠀⠈⣻⣾⣧⣀⠀⠀⠁⠉⠈⠀⢊⣐⣂⠀⢁⣴⠆⠀⠀⠈⣳⣶⣆⡀⠀⠀⠁⠁⠀⠀⎥
⎢⠀⠀⠲⠆⣐⡒⠀⠀⠀⠀⠉⢻⣿⣿⣦⡄⠀⠀⠀⠀⠀⠀⠤⠀⣐⠂⠀⠀⠀⠀⠙⢯⣯⣿⣤⠀⠀⠀⠀⠀⎥
⎢⠸⠄⠈⣿⡶⠀⠠⠄⠀⠀⠀⠀⠈⠿⠻⣦⣤⣀⠀⠀⠥⠀⠉⣷⡦⠀⠠⠄⠀⠀⠀⠀⠘⠿⠷⣤⣤⡀⠀⠀⎥
⎢⠈⠁⠀⠈⠁⠀⠘⣓⠀⢨⡅⠀⠀⠀⠀⢻⣿⣿⣦⡀⠈⠀⠀⠈⠁⠀⠈⣁⠀⠬⠅⠀⠀⠀⠀⠿⣿⣿⣦⡀⎥
⎢⠀⢲⠀⠀⠀⠀⠀⠲⣶⣒⠂⠀⠀⠀⠀⠀⠈⠻⣿⣿⠀⢂⠀⠀⠀⠀⠀⠲⢦⣒⠀⠀⠀⠀⠀⠀⠘⠻⣿⣿⎥
⎢⡙⢆⣁⠃⠒⠰⢘⠠⢄⡀⢊⢰⠀⠀⠁⠃⠂⠀⠠⢀⡻⢎⡈⠓⠂⠶⡙⠠⢄⡀⡃⣆⠀⠈⠐⠃⠂⠀⠠⢀⎥
⎢⠰⡀⠙⣧⣄⠀⠠⠀⠀⠀⠈⠘⠀⠃⢧⣤⡀⠀⠀⠀⢦⠈⠻⣦⡀⠀⠠⠀⠀⠀⠁⠛⠘⠸⣤⣤⡀⠀⠀⠀⎥
⎢⠈⡅⠀⠈⠙⢶⡄⠀⠀⠀⢁⣴⠰⠘⠈⠋⠁⠀⠀⠀⢨⡄⠀⠈⠻⣦⡀⠀⠀⠀⢡⡆⠆⠘⠙⠈⠁⠀⠀⠀⎥
⎢⠑⡀⠁⠂⠀⠀⠹⣦⡀⠀⠈⠁⠀⠀⠀⠆⠆⢠⢠⡀⠓⡈⠀⠂⠀⠈⠱⣦⡀⠀⠉⠀⠀⠀⠀⠆⠆⢠⢠⡀⎥
⎢⠀⠱⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⡀⡄⢨⢳⠀⠱⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⡀⡄⣬⢳⎥
⎢⠉⠄⣭⠀⠁⠴⠟⠀⠀⠈⢹⣾⡷⣄⠀⠀⠁⠁⠀⠀⠩⢬⣥⠀⠡⠶⠃⠀⠀⠈⣿⣿⣦⡄⠀⠀⠁⠁⠀⠀⎥
⎢⢀⠀⢐⣂⢈⡉⠀⠀⠀⠀⠈⠹⣯⣿⣶⡄⠀⠀⠀⠀⡀⠀⣒⡀⣈⠁⠀⠀⠀⠀⠈⠿⣿⣿⣶⠀⠀⠀⠀⠀⎥
⎢⠠⠂⠀⣽⡓⠀⠠⠄⠀⠀⠀⠀⠀⠛⠙⣧⣤⡄⠀⠀⠴⠀⠀⣿⡓⠀⠠⠄⠀⠀⠀⠀⠘⠛⠛⣤⣤⡄⠀⠀⎥
⎢⠈⠀⠀⠈⠁⠀⠈⣁⠀⠨⠅⠀⠀⠀⠀⠻⣿⣿⣶⡀⠈⠀⠀⠈⠁⠀⠈⣁⠀⠬⠅⠀⠀⠀⠀⠿⣿⣿⣶⡀⎥
⎣⠀⢂⠀⠀⠀⠀⠀⠒⢦⣛⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⠀⢂⠀⠀⠀⠀⠀⠲⢦⣛⠀⠀⠀⠀⠀⠀⠘⠻⣿⣿⎦

In [19]:
A_d = (kron(tξ,tη) - kron(tη,tξ))' / 2
B_d = A_d ⋅ Grad(U) + n' ⋅ Φ

α = 1e-4
Kd = ∫(B_d' ⋅ [α * b * mat.μ;;] ⋅ B_d; Γ="plates")
Kd[:,:]

396×396 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4527 stored entries:
⎡⠱⢆⠑⠂⠖⠤⠘⢂⡀⡀⠏⠄⠀⠀⠒⠂⠆⠀⢀⣀⠳⣄⠑⠲⠂⠤⠘⢀⢀⡰⠁⠄⠀⠀⠒⠲⠀⠀⢀⢀⎤
⎢⠱⠀⠻⣦⡀⠀⠨⠄⠀⠀⠉⠘⠸⠀⣆⣤⡀⠀⠀⠀⠵⠀⠻⣦⡀⠀⠨⠀⠀⠀⠁⠘⠸⠀⣆⣄⠀⠀⠀⠀⎥
⎢⠘⡅⠀⠈⠻⣦⡀⠀⠀⠀⢁⡄⠰⠸⠘⠋⠁⠀⠀⠀⠘⡄⠀⠉⠻⣦⡀⠀⠀⠈⢀⡄⠰⠸⠘⠉⠀⠀⠀⠀⎥
⎢⠲⢀⠂⠆⠀⠈⠻⣦⡀⠀⠛⠁⠀⠀⠀⠆⢰⢠⢀⡀⠆⡀⠂⠆⠀⠈⢻⣄⠀⠀⠛⠁⠀⠀⠀⠆⡆⡄⢀⠀⎥
⎢⠀⠨⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⡀⣀⢸⢻⠀⠡⠀⠀⠀⠀⠀⠘⠷⣄⠀⠀⠀⠀⠀⢀⠀⡀⡜⢷⎥
⎢⠋⠅⣃⠀⠁⠴⠟⠀⠀⠈⣻⣾⣧⣀⠀⠀⠁⠉⠈⠀⠋⠄⣃⠈⠀⠴⠟⠀⠀⠉⣺⣾⣦⡀⠀⠈⠀⠁⠁⠀⎥
⎢⠀⠀⠒⠂⣐⡂⠀⠀⠀⠀⠉⢻⣿⣿⣦⡄⠀⠀⠀⠀⠀⠀⠒⠀⣐⡂⠀⠀⠀⠀⠉⢻⣿⣿⣦⠀⠀⠀⠀⠀⎥
⎢⠸⠀⠈⣽⡶⠀⠠⠄⠀⠀⠀⠀⠈⠿⠻⣦⣠⣀⠀⠀⠼⠀⠈⣽⠶⠀⠠⠀⠀⠀⠀⠀⠈⠷⠻⣄⣤⡀⠀⠀⎥
⎢⠈⠁⠀⠈⠁⠀⠐⣒⠀⢨⡅⠀⠀⠀⠀⢺⣿⣿⣦⡀⠈⠀⠀⠉⠀⠀⢐⣂⠀⣬⠀⠀⠀⠀⠀⢿⣺⣷⣆⠀⎥
⎢⠀⢰⠀⠀⠀⠀⠀⠰⣶⣒⠂⠀⠀⠀⠀⠀⠈⠻⣿⣿⠀⢆⠀⠀⠀⠀⠀⠶⣆⡒⠀⠀⠀⠀⠀⠀⠙⢿⣿⣟⎥
⎢⠙⢦⠑⠃⠒⠤⠈⠡⠄⡀⠋⠄⠀⠀⠒⠃⠂⠀⠠⢄⠑⢄⠑⠓⠂⠤⠈⠠⢀⡐⠁⠄⠀⠀⠒⠑⠀⠀⠠⢀⎥
⎢⢱⡀⠻⣦⡄⠀⠨⠄⠀⠀⡉⠘⠘⠀⣆⣤⡄⠀⠀⠀⢵⠀⠻⣦⡀⠀⠨⠀⠀⢀⠁⠘⠘⠀⣆⣤⠀⠀⠀⠀⎥
⎢⠈⡄⠀⠈⠻⣦⡀⠀⠀⠀⢀⡄⠰⠸⠘⠃⠀⠀⠀⠀⠈⡄⠀⠈⠻⣦⡀⠀⠀⠀⢀⡄⠰⠸⠘⠀⠀⠀⠀⠀⎥
⎢⠒⢀⠂⠂⠀⠈⠛⢶⣀⠀⠛⠁⠀⠀⠀⠂⠰⢰⢠⡄⠂⡀⠂⠂⠀⠈⠻⣦⡀⠀⠛⠁⠀⠀⠀⠂⠆⡆⢠⠀⎥
⎢⢀⡰⠀⠀⡀⠀⠀⠀⠙⢧⡄⠀⠀⠀⠀⠀⡀⣤⢨⠹⢀⠰⠀⢀⠀⠀⠀⠈⠻⣦⠀⠀⠀⠀⠀⢀⠀⡄⡌⠻⎥
⎢⠁⠄⣁⠀⠀⠴⠟⠀⠀⠀⣺⣾⣧⣀⠀⠀⠀⠀⠀⠀⠁⠄⣁⠀⠀⠴⠟⠀⠀⠀⣻⣾⣦⡀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠒⠂⣐⡂⠀⠀⠀⠀⠈⠻⣿⣿⢦⡄⠀⠀⠀⠀⠀⠀⠒⠀⣐⡂⠀⠀⠀⠀⠈⠻⣿⣿⢦⠀⠀⠀⠀⠀⎥
⎢⢸⡀⠈⢽⡖⠀⠠⠄⠀⢀⡀⠀⠈⠛⠛⢦⣤⣄⠀⠀⢜⠀⠈⣽⠒⠀⠠⠀⠀⢀⠀⠀⠈⠓⠛⣤⢤⡀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠈⠭⠀⠠⠄⠀⠀⠀⠀⠻⢾⣾⣷⣄⠀⠀⠀⠀⠀⠀⠨⠥⠀⠤⠀⠀⠀⠀⠀⠳⣿⣿⣦⡀⎥
⎣⠀⢐⠀⠀⠀⠀⠀⠐⢶⣍⠁⠀⠀⠀⠀⠀⠈⠙⣿⢿⠀⢂⠀⠀⠀⠀⠀⠒⣦⡉⠀⠀⠀⠀⠀⠀⠈⠻⡿⣯⎦

In [20]:
fu = ∫(U ⋅ [0.0, 0.0, 1.0]; Γ="load")
fφ = ∫(Φ ⋅ [0.0, 1.0, 0.0]; Γ="load")
f = SystemVector([fu, fφ]);

bc_u = BoundaryCondition("clamp"; field=U, ux=0, uy=0, uz=0)
bc_φ = BoundaryCondition("clamp"; field=Φ, φx=0, φy=0, φz=0);

In [21]:
u, φ = solveField(K + Kd, f, support=[bc_u, bc_φ]);

In [22]:
showDoFResults(u, name="u", visible=true, factor=10);

In [23]:
openPostProcessor()